# 🧠 Notebook 14 — Slaying the Memory Monster
### Group Relative Policy Optimization (GRPO)

**Series**: RL Notebook Series · Act V — LLM Alignment · Post 14 of 17

---

## From Games to Language

We've spent 12 notebooks teaching agents to play games. But what about teaching an AI to **write**?

That's exactly what **RLHF** (Reinforcement Learning from Human Feedback) does for ChatGPT, Claude, and every modern LLM.
The pipeline has 3 stages:

| Stage | What happens | Analogy |
| :--- | :--- | :--- |
| **1. SFT (Supervised Fine-Tuning)** | Pre-train the model on good text | Teaching a student from textbooks |
| **2. Reward Model** | Train a scorer that rates output quality | Hiring a teacher to grade essays |
| **3. RL Fine-Tuning** | Use PPO to maximize the reward score | Student rewrites essays to get better grades |

Stage 3 is where our RL knowledge comes in! But there's a problem...

### 🐘 The Memory Monster

PPO needs a **Critic** (value network) that is the **same size as the Actor** (the LLM itself).
For a 7-billion parameter LLM, that means storing **14 billion parameters** in GPU memory. Ouch!

DeepSeek's **GRPO** (Group Relative Policy Optimization) kills the Critic entirely.
Instead of training a value network to estimate advantages, it generates a **group of outputs**,
scores them all, and uses the **relative ranking within the group** as the advantage.

Let's build it from scratch — using a tiny language model that generates baby names.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})
torch.manual_seed(42)
np.random.seed(42)

## 1. Our Tiny Language Model (The Karpathy Special)

Inspired by Andrej Karpathy's famous `makemore`, we'll build a character-level language model.
It predicts the next character given the previous characters. Dead simple — and it's a **real language model**!

We'll train it on baby names so it learns to generate plausible-sounding new names.
This is our **SFT (Supervised Fine-Tuning) phase** — Stage 1 of the RLHF pipeline.

In [ ]:
# A curated list of baby names for training
NAMES = [
    'emma', 'olivia', 'ava', 'sophia', 'isabella', 'mia', 'charlotte', 'amelia',
    'harper', 'evelyn', 'abigail', 'emily', 'ella', 'elizabeth', 'camila', 'luna',
    'sofia', 'avery', 'mila', 'aria', 'scarlett', 'penelope', 'layla', 'chloe',
    'victoria', 'madison', 'eleanor', 'grace', 'nora', 'riley', 'zoey', 'hannah',
    'hazel', 'lily', 'ellie', 'violet', 'lillian', 'zoe', 'stella', 'aurora',
    'natalie', 'emilia', 'everly', 'leah', 'aubrey', 'willow', 'addison', 'lucy',
    'audrey', 'bella', 'nova', 'brooklyn', 'paisley', 'savannah', 'claire', 'skylar',
    'liam', 'noah', 'oliver', 'james', 'elijah', 'william', 'henry', 'lucas',
    'benjamin', 'theodore', 'jack', 'levi', 'alexander', 'mason', 'ethan', 'daniel',
    'jacob', 'michael', 'logan', 'jackson', 'sebastian', 'aiden', 'owen', 'samuel',
    'ryan', 'nathan', 'caleb', 'christian', 'dylan', 'landon', 'josiah', 'andrew',
    'thomas', 'gabriel', 'anthony', 'leo', 'lincoln', 'jaxon', 'asher', 'mateo',
    'oscar', 'ezra', 'hudson', 'miles', 'elias', 'kai', 'ace', 'nova',
    'iris', 'ivy', 'ada', 'eve', 'axel', 'cole', 'finn', 'dean',
    'jade', 'ruby', 'hope', 'eden', 'kira', 'maya', 'sara', 'alba',
    'felix', 'rex', 'max', 'hugo', 'lars', 'otto', 'rome', 'sage',
]

# Build character vocabulary
# '.' is the special start/end token (like Karpathy's makemore)
chars = sorted(list(set(''.join(NAMES))))
VOCAB = ['.'] + chars  # '.' = start/end token
VOCAB_SIZE = len(VOCAB)
stoi = {ch: i for i, ch in enumerate(VOCAB)}  # string to index
itos = {i: ch for ch, i in stoi.items()}       # index to string

print(f"Vocabulary ({VOCAB_SIZE} chars): {VOCAB}")
print(f"Example: 'emma' → {[stoi[c] for c in '.emma.']}")

### Building the Training Dataset

We convert each name into (context, target) pairs.
With a context window of 3 characters, the name `emma` becomes:

| Context | Target |
| :--- | :--- |
| `...` | `e` |
| `..e` | `m` |
| `.em` | `m` |
| `emm` | `a` |
| `mma` | `.` (end) |

In [ ]:
CONTEXT_LEN = 3  # How many previous characters the model sees

def build_dataset(names):
    """Convert names into (context, target) training pairs."""
    xs, ys = [], []
    for name in names:
        # Pad the name with start/end tokens
        padded = '.' * CONTEXT_LEN + name + '.'
        for i in range(len(padded) - CONTEXT_LEN):
            context = [stoi[c] for c in padded[i:i+CONTEXT_LEN]]
            target = stoi[padded[i+CONTEXT_LEN]]
            xs.append(context)
            ys.append(target)
    return torch.tensor(xs), torch.tensor(ys)

X_train, Y_train = build_dataset(NAMES)
print(f"Training set: {X_train.shape[0]} examples")
print(f"Example: context={X_train[0].tolist()} → target={Y_train[0].item()} ('{itos[Y_train[0].item()]}')") 

### The Character-Level Model

A simple embedding + MLP, exactly like Karpathy's `makemore` Part 2.
Each character is embedded into a small vector, then the concatenated context embeddings
are fed through a hidden layer to predict the probability of the next character.

In [ ]:
class CharLM(nn.Module):
    """A tiny character-level language model (Karpathy's makemore style)."""
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=64, context_len=CONTEXT_LEN):
        super().__init__()
        self.context_len = context_len
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * context_len, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        """x: (batch, context_len) of token indices → logits (batch, vocab_size)"""
        emb = self.embed(x)                # (batch, context_len, embed_dim)
        emb = emb.view(emb.shape[0], -1)   # (batch, context_len * embed_dim)
        h = torch.tanh(self.fc1(emb))      # (batch, hidden_dim)
        logits = self.fc2(h)               # (batch, vocab_size)
        return logits
    
    def get_probs(self, x):
        """Return probabilities (softmax of logits)."""
        return F.softmax(self.forward(x), dim=-1)

print(f"Model parameters: {sum(p.numel() for p in CharLM(VOCAB_SIZE).parameters()):,}")

### SFT Pre-Training

This is Stage 1 of the RLHF pipeline. We train the model to predict the next character
using standard cross-entropy loss — exactly like training any language model.

In [ ]:
def pretrain_sft(model, X, Y, epochs=200, lr=0.01):
    """Supervised Fine-Tuning (SFT) — standard next-token prediction."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        logits = model(X)
        loss = F.cross_entropy(logits, Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")
    return losses

sft_model = CharLM(VOCAB_SIZE)
print("Pre-training the SFT model...")
sft_losses = pretrain_sft(sft_model, X_train, Y_train)

plt.plot(sft_losses, color='#3b82f6')
plt.title("SFT Pre-Training Loss"); plt.xlabel("Epoch"); plt.ylabel("Cross-Entropy Loss")
plt.show()

### Generating Names

Let's see what our SFT model produces! We start with the padding token `...` and
sample one character at a time until we hit the end token `.`

In [ ]:
@torch.no_grad()
def generate_name(model, max_len=10):
    """Generate a single name by sampling from the model character-by-character."""
    context = [0] * CONTEXT_LEN  # Start with '...' padding
    name = []
    for _ in range(max_len):
        x = torch.tensor([context])
        probs = model.get_probs(x)
        ix = torch.multinomial(probs[0], 1).item()
        if ix == 0:  # End token
            break
        name.append(itos[ix])
        context = context[1:] + [ix]  # Slide the context window
    return ''.join(name)

print("📝 SFT-generated names:")
for _ in range(15):
    print(f"  {generate_name(sft_model)}")

## 2. The Reward Function (Our "Reward Model")

In a real RLHF pipeline, a separate neural network would score text quality.
For our toy example, we use a deterministic scoring function that grades names
based on simple quality heuristics.

This lets us objectively measure whether alignment is working!

In [ ]:
VOWELS = set('aeiou')
CONSONANTS = set('bcdfghjklmnpqrstvwxyz')

def reward_name(name):
    """Score a generated name. Higher = better quality."""
    if len(name) == 0:
        return -2.0
    
    score = 0.0
    
    # 1. Length: prefer 3-7 characters
    if 3 <= len(name) <= 7:
        score += 1.0
    elif len(name) < 3:
        score -= 1.0
    elif len(name) > 8:
        score -= 0.5
    
    # 2. No consecutive repeated characters ('aaa' is ugly)
    has_repeats = any(name[i] == name[i+1] == name[i+2] for i in range(len(name)-2)) if len(name) >= 3 else False
    if has_repeats:
        score -= 1.5
    
    # 3. Nice vowel-consonant alternation
    if len(name) >= 2:
        alternations = sum(
            1 for i in range(len(name)-1)
            if (name[i] in VOWELS) != (name[i+1] in VOWELS)
        )
        score += 0.5 * (alternations / (len(name) - 1))
    
    # 4. Ends in a vowel (common in many names)
    if name[-1] in VOWELS:
        score += 0.5
    
    # 5. Starts with a consonant
    if name[0] in CONSONANTS:
        score += 0.3
    
    return score

# Test the reward function on some names
test_names = ['emma', 'zz', 'kira', 'aaaaaa', 'sophia', 'rx', 'luna', 'bb']
for n in test_names:
    print(f"  '{n}' → reward = {reward_name(n):.2f}")

## 2b. Training a Learned Reward Model (Stage 2)

In a real RLHF pipeline, you don't have a handwritten reward function.
Instead, you collect **human preferences** — pairs of outputs where a human picks the better one —
and train a **neural network** to predict which output is preferred.

This is the **Bradley-Terry model**: given two outputs, the probability that output A beats output B is:

$$P(A \succ B) = \sigma(r(A) - r(B)) = \frac{1}{1 + e^{-(r(A) - r(B))}}$$

where $r(\cdot)$ is the learned reward. Let's build one!

In [ ]:
class RewardModel(nn.Module):
    """A tiny learned reward model — predicts quality of a name."""
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32, max_len=10):
        super().__init__()
        self.max_len = max_len
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * max_len, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)  # Outputs a scalar reward
    
    def forward(self, token_ids):
        """token_ids: (batch, max_len) padded token indices → scalar reward."""
        emb = self.embed(token_ids)               # (batch, max_len, embed_dim)
        emb = emb.view(emb.shape[0], -1)          # (batch, max_len * embed_dim)
        h = torch.tanh(self.fc1(emb))
        return self.fc2(h).squeeze(-1)             # (batch,)

def name_to_tensor(name, max_len=10):
    """Convert a name string to a padded tensor of token indices."""
    tokens = [stoi[c] for c in name[:max_len]]
    tokens += [0] * (max_len - len(tokens))  # Pad with end tokens
    return torch.tensor(tokens)

print(f"Reward Model parameters: {sum(p.numel() for p in RewardModel(VOCAB_SIZE).parameters()):,}")

### Collecting Preferences & Training the RM

We generate pairs of names from the SFT model, score them with our handcrafted reward,
and pretend these are *human preferences*. Then we train the RM to reproduce these preferences.

In [ ]:
# Generate preference pairs using our handcrafted reward as a stand-in for humans
def generate_preference_pairs(model, num_pairs=500, group_size=6):
    """Generate (winner, loser) pairs for reward model training."""
    winners, losers = [], []
    for _ in range(num_pairs):
        names = [generate_name(model) for _ in range(group_size)]
        names = [n for n in names if len(n) > 0]
        if len(names) < 2:
            continue
        scored = [(n, reward_name(n)) for n in names]
        scored.sort(key=lambda x: x[1])
        if scored[-1][1] > scored[0][1]:
            winners.append(scored[-1][0])
            losers.append(scored[0][0])
    return winners, losers

pref_winners, pref_losers = generate_preference_pairs(sft_model)
print(f"Collected {len(pref_winners)} preference pairs")

# Train the reward model with Bradley-Terry loss (batched for stability)
rm = RewardModel(VOCAB_SIZE)
rm_opt = optim.Adam(rm.parameters(), lr=5e-4)
BATCH_SIZE = 32

rm_losses = []
for epoch in range(200):
    total_loss = 0
    indices = np.random.permutation(len(pref_winners))
    n_batches = 0
    for start in range(0, len(indices), BATCH_SIZE):
        batch_idx = indices[start:start+BATCH_SIZE]
        w_tensors = torch.stack([name_to_tensor(pref_winners[i]) for i in batch_idx])
        l_tensors = torch.stack([name_to_tensor(pref_losers[i]) for i in batch_idx])
        
        r_w = rm(w_tensors)  # Rewards for winners
        r_l = rm(l_tensors)  # Rewards for losers
        
        # Bradley-Terry: P(winner > loser) = sigmoid(r_w - r_l)
        loss = -F.logsigmoid(r_w - r_l).mean()
        
        rm_opt.zero_grad()
        loss.backward()
        rm_opt.step()
        total_loss += loss.item()
        n_batches += 1
    
    rm_losses.append(total_loss / n_batches)
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1:3d} | RM Loss: {rm_losses[-1]:.4f}")

plt.plot(rm_losses, color='#f59e0b')
plt.title('Reward Model Training (Bradley-Terry Loss)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.show()

In [ ]:
# Test: does the learned RM prefer the same names as our handcrafted reward?
# The raw RM scores won't match — the RM only learns to RANK, not to output
# the same numbers. So we check if the RM correctly picks the winner in new pairs.
print("Pairwise accuracy test: does the RM pick the same winner as the handcrafted reward?")

correct = 0
total = 0
with torch.no_grad():
    for _ in range(200):
        n1, n2 = generate_name(sft_model), generate_name(sft_model)
        if len(n1) == 0 or len(n2) == 0:
            continue
        # Handcrafted preference
        hand_prefers_1 = reward_name(n1) > reward_name(n2)
        # Learned RM preference
        r1 = rm(name_to_tensor(n1).unsqueeze(0)).item()
        r2 = rm(name_to_tensor(n2).unsqueeze(0)).item()
        rm_prefers_1 = r1 > r2
        if hand_prefers_1 == rm_prefers_1:
            correct += 1
        total += 1

accuracy = correct / total * 100
print(f"\n✅ Pairwise accuracy: {correct}/{total} = {accuracy:.1f}%")
print(f"   (50% = random, 100% = perfect agreement)")
print(f"\n👆 The RM learned to RANK names similarly to our handcrafted function,")
print(f"   even though the raw scores differ. That's all we need!")
print(f"\n   In practice, we'd use this learned RM for GRPO/PPO training.")
print(f"   For clarity, we'll stick with the handcrafted version below.")

## 3. Generating Names with Log-Probabilities (RL Mode)

For RL training, we need to track the **log-probability of every character choice**.
This is the same concept from Notebooks 7-9 (REINFORCE, Actor-Critic, PPO) —
just applied to text generation instead of game actions.

In [ ]:
def generate_name_with_logprobs(model, max_len=10):
    """Generate a name and return the sequence of (token, log_prob) tuples."""
    context = [0] * CONTEXT_LEN
    tokens = []
    log_probs = []
    
    for _ in range(max_len):
        x = torch.tensor([context])
        logits = model(x)
        probs = F.softmax(logits, dim=-1)
        dist = Categorical(probs[0])
        
        ix = dist.sample()
        log_prob = dist.log_prob(ix)
        
        if ix.item() == 0:  # End token
            break
        
        tokens.append(ix.item())
        log_probs.append(log_prob)
        context = context[1:] + [ix.item()]
    
    name = ''.join(itos[t] for t in tokens)
    return name, tokens, log_probs

# Quick test
name, toks, lps = generate_name_with_logprobs(sft_model)
print(f"Generated: '{name}'")
print(f"Tokens: {toks}")
print(f"Log probs: {[f'{lp.item():.3f}' for lp in lps]}")
print(f"Total log prob: {sum(lp.item() for lp in lps):.3f}")

## 4. PPO: The Expensive Way (with a Critic)

Let's first implement standard PPO, the same algorithm used to train ChatGPT.
This requires **two models**:
- **Actor** = our name generator (policy)
- **Critic** = a value network that estimates how good a generation will be

Notice the memory cost: we need to store the parameters for **both** networks!

In [ ]:
class CriticNetwork(nn.Module):
    """Value network that estimates the expected reward from the start token."""
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=64, context_len=CONTEXT_LEN):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * context_len, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)  # Outputs a single scalar value
    
    def forward(self, x):
        emb = self.embed(x).view(x.shape[0], -1)
        h = torch.tanh(self.fc1(emb))
        return self.fc2(h)

actor_params = sum(p.numel() for p in CharLM(VOCAB_SIZE).parameters())
critic_params = sum(p.numel() for p in CriticNetwork(VOCAB_SIZE).parameters())
print(f"🐘 PPO Memory Cost:")
print(f"   Actor:  {actor_params:,} parameters")
print(f"   Critic: {critic_params:,} parameters")
print(f"   Total:  {actor_params + critic_params:,} parameters")
print(f"\n   (For GPT-4 scale, imagine this is 2 × 1.8 TRILLION parameters!)")

In [ ]:
def train_ppo(sft_model, num_episodes=80, lr=1e-4, gamma=0.99, 
              eps_clip=0.2, K_epochs=2, batch_size=16, kl_coeff=2.0):
    """Standard PPO with a full Critic network."""
    # Clone the SFT model as our Actor
    actor = CharLM(VOCAB_SIZE)
    actor.load_state_dict(sft_model.state_dict())
    
    # Freeze a reference copy for KL penalty
    ref_model = CharLM(VOCAB_SIZE)
    ref_model.load_state_dict(sft_model.state_dict())
    for p in ref_model.parameters():
        p.requires_grad = False
    
    # Create the Critic (the memory monster!)
    critic = CriticNetwork(VOCAB_SIZE)
    
    actor_opt = optim.Adam(actor.parameters(), lr=lr)
    critic_opt = optim.Adam(critic.parameters(), lr=lr)
    
    reward_history = []
    
    for episode in range(num_episodes):
        # Collect a batch of rollouts
        batch_rewards = []
        batch_log_probs = []
        batch_names = []
        
        for _ in range(batch_size):
            name, tokens, log_probs = generate_name_with_logprobs(actor)
            reward = reward_name(name)
            batch_rewards.append(reward)
            batch_log_probs.append(log_probs)
            batch_names.append(name)
        
        avg_reward = np.mean(batch_rewards)
        reward_history.append(avg_reward)
        
        # Get Critic's baseline estimate
        start_context = torch.tensor([[0] * CONTEXT_LEN])  # '...' start token
        baseline = critic(start_context).item()
        
        # PPO Update
        for k in range(K_epochs):
            actor_loss_total = 0
            critic_loss_total = 0
            
            for i in range(batch_size):
                if len(batch_log_probs[i]) == 0:
                    continue
                
                reward = batch_rewards[i]
                advantage = reward - baseline  # Simple advantage: reward - value
                
                # Recompute current log probs (for PPO ratio)
                old_log_prob = torch.stack(batch_log_probs[i]).detach().sum()
                
                # Re-generate log probs under current policy by replaying the tokens
                name = batch_names[i]
                padded = '.' * CONTEXT_LEN + name + '.'
                new_log_prob = torch.tensor(0.0)
                ref_log_prob = torch.tensor(0.0)
                
                for j in range(len(name)):
                    ctx = [stoi[c] for c in padded[j:j+CONTEXT_LEN]]
                    target = stoi[padded[j+CONTEXT_LEN]]
                    x = torch.tensor([ctx])
                    
                    logits = actor(x)
                    log_p = F.log_softmax(logits, dim=-1)[0, target]
                    new_log_prob = new_log_prob + log_p
                    
                    with torch.no_grad():
                        ref_logits = ref_model(x)
                        ref_log_p = F.log_softmax(ref_logits, dim=-1)[0, target]
                        ref_log_prob = ref_log_prob + ref_log_p
                
                # PPO clipped surrogate
                ratio = torch.exp(new_log_prob - old_log_prob)
                surr1 = ratio * advantage
                surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * advantage
                
                # KL divergence penalty (keep policy close to SFT)
                kl = (new_log_prob - ref_log_prob).detach().abs()
                
                actor_loss_total += -torch.min(surr1, surr2) + kl_coeff * kl
                critic_loss_total += (reward - critic(start_context)).pow(2)
            
            actor_loss = actor_loss_total / batch_size
            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()
            
            critic_loss = critic_loss_total / batch_size
            critic_opt.zero_grad()
            critic_loss.backward()
            critic_opt.step()
        
        if (episode + 1) % 20 == 0:
            print(f"  Episode {episode+1:3d} | Avg Reward: {avg_reward:.3f} | Baseline: {baseline:.3f}")
    
    return actor, reward_history

print("Training PPO (with Critic)...")
ppo_actor, ppo_rewards = train_ppo(sft_model)

In [ ]:
print("\n📝 PPO-generated names:")
ppo_names = [generate_name(ppo_actor) for _ in range(15)]
for n in ppo_names:
    print(f"  '{n}' (reward: {reward_name(n):.2f})")

## 5. GRPO: Kill the Critic! 🗡️

Here comes the magic. DeepSeek's insight:

> *"Why train a Critic to estimate the value when you can just... measure it directly?"*

**GRPO Algorithm:**
1. For each prompt, generate **G** different outputs (a "group")
2. Score each output with the reward function
3. Compute the **group-relative advantage**: how much better/worse each output is compared to the group average
4. Use the standard PPO clipped objective with these advantages

$$\hat{A}_i = \frac{r_i - \mu_{group}}{\sigma_{group} + \epsilon}$$

No Critic. No value network. **Half the parameters!** 🎉

### Why does this work?
Think of it like grading on a curve. You don't need to know what an "A" is worth in absolute terms.
You just need to know that within your group, some answers are better than others.
The Critic was trying to estimate the average — but with GRPO, we just **compute** the average directly.

In [ ]:
def sample_group(model, group_size=8, max_len=10):
    """Sample a group of names from the model, each with their log probabilities."""
    # ============================================================
    # TODO 1: Sample `group_size` names from the model.
    #
    # For each name in the group:
    #   - Use generate_name_with_logprobs(model) to get (name, tokens, log_probs)
    #   - Collect all names in a list
    #   - Collect all log_probs lists in another list
    # Return: (names, all_log_probs)
    # ============================================================
    names = []
    all_log_probs = []
    
    raise NotImplementedError("Sample a group of names!")
    
    return names, all_log_probs

def compute_grpo_advantages(rewards):
    """Compute group-relative advantages: normalize rewards within the group."""
    # ============================================================
    # TODO 2: Compute the group-relative advantage for each reward.
    #
    # This is the core GRPO insight that replaces the Critic!
    # Given a list of rewards from the group, compute:
    #   advantage_i = (reward_i - group_mean) / (group_std + epsilon)
    # ============================================================
    rewards = np.array(rewards)
    advantages = None
    
    raise NotImplementedError("Compute the group-relative advantages!")
    
    return advantages

# Quick demo
demo_names, _ = sample_group(sft_model, group_size=8)
demo_rewards = [reward_name(n) for n in demo_names]
demo_advantages = compute_grpo_advantages(demo_rewards)

print("Group-Relative Advantage Demo:")
print(f"{'Name':<12} {'Reward':>7} {'Advantage':>10}")
print("-" * 32)
for n, r, a in zip(demo_names, demo_rewards, demo_advantages):
    emoji = '✅' if a > 0 else '❌'
    print(f"{emoji} {n:<10} {r:>7.2f} {a:>10.2f}")
print(f"\nGroup mean: {np.mean(demo_rewards):.2f}, Group std: {np.std(demo_rewards):.2f}")

<details>
<summary>🔍 <b>Hint for TODO 1</b> (click to reveal)</summary>

This is a simple loop! Call `generate_name_with_logprobs(model)` inside a `for` loop
`group_size` times. Append the `name` to one list and `log_probs` to another.

</details>

<details>
<summary>🔍 <b>Hint for TODO 2</b> (click to reveal)</summary>

Convert rewards to a numpy array, then:
```python
mean = rewards.mean()
std = rewards.std() + 1e-8
advantages = (rewards - mean) / std
```
This is just z-score normalization! The `1e-8` prevents division by zero.

</details>

In [ ]:
def train_grpo(sft_model, num_episodes=80, lr=1e-4, 
               eps_clip=0.2, K_epochs=2, group_size=8, kl_coeff=2.0):
    """GRPO — PPO without the Critic. Uses group-relative advantages instead!"""
    # Clone the SFT model as our policy
    policy = CharLM(VOCAB_SIZE)
    policy.load_state_dict(sft_model.state_dict())
    
    # Frozen reference model for KL penalty
    ref_model = CharLM(VOCAB_SIZE)
    ref_model.load_state_dict(sft_model.state_dict())
    for p in ref_model.parameters():
        p.requires_grad = False
    
    # NO CRITIC! Just one optimizer for the policy!
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    reward_history = []
    
    for episode in range(num_episodes):
        # Step 1: Sample a GROUP of names
        names, old_log_probs_list = sample_group(policy, group_size=group_size)
        
        # Step 2: Score them all
        rewards = [reward_name(n) for n in names]
        
        # Step 3: Compute GROUP-RELATIVE advantages (THE GRPO MAGIC!)
        advantages = compute_grpo_advantages(rewards)
        
        avg_reward = np.mean(rewards)
        reward_history.append(avg_reward)
        
        # Step 4: PPO update (same clipped surrogate, but NO critic loss!)
        for k in range(K_epochs):
            policy_loss_total = 0
            valid_samples = 0
            
            for i in range(group_size):
                if len(old_log_probs_list[i]) == 0:
                    continue
                
                advantage = advantages[i]
                old_log_prob = torch.stack(old_log_probs_list[i]).detach().sum()
                
                # Replay the name to get current log probs
                name = names[i]
                padded = '.' * CONTEXT_LEN + name + '.'
                new_log_prob = torch.tensor(0.0)
                ref_log_prob = torch.tensor(0.0)
                
                for j in range(len(name)):
                    ctx = [stoi[c] for c in padded[j:j+CONTEXT_LEN]]
                    target = stoi[padded[j+CONTEXT_LEN]]
                    x = torch.tensor([ctx])
                    
                    logits = policy(x)
                    log_p = F.log_softmax(logits, dim=-1)[0, target]
                    new_log_prob = new_log_prob + log_p
                    
                    with torch.no_grad():
                        ref_logits = ref_model(x)
                        ref_log_p = F.log_softmax(ref_logits, dim=-1)[0, target]
                        ref_log_prob = ref_log_prob + ref_log_p
                
                # ============================================================
                # TODO 3: Compute the GRPO loss for this sample.
                #
                # 1. Compute the probability ratio: exp(new_log_prob - old_log_prob)
                # 2. Compute surr1 = ratio * advantage
                # 3. Compute surr2 = clamp(ratio, 1-eps, 1+eps) * advantage
                # 4. Compute the KL penalty between the policy and reference model
                # 5. Loss = -min(surr1, surr2) + kl_coeff * kl
                #
                # This is the same PPO loss from Notebook 09 — but without
                # any Critic loss! The advantage came from the GROUP, not a Critic.
                # ============================================================
                sample_loss = None
                
                raise NotImplementedError("Implement the GRPO loss!")
                
                policy_loss_total += sample_loss
                valid_samples += 1
            
            if valid_samples > 0:
                loss = policy_loss_total / valid_samples
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                optimizer.step()
        
        if (episode + 1) % 20 == 0:
            print(f"  Episode {episode+1:3d} | Avg Reward: {avg_reward:.3f}")
    
    return policy, reward_history

grpo_params = sum(p.numel() for p in CharLM(VOCAB_SIZE).parameters())
print(f"⚡ GRPO Memory Cost:")
print(f"   Policy: {grpo_params:,} parameters")
print(f"   Critic: 0 parameters (KILLED!)")
print(f"   Total:  {grpo_params:,} parameters\n")

print("Training GRPO (no Critic)...")
grpo_policy, grpo_rewards = train_grpo(sft_model)

<details>
<summary>🔍 <b>Hint for TODO 3</b> (click to reveal)</summary>

This is the exact same PPO clipped surrogate loss from Notebook 09:
```python
ratio = torch.exp(new_log_prob - old_log_prob)
surr1 = ratio * advantage
surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * advantage
kl = (new_log_prob - ref_log_prob).detach().abs()
sample_loss = -torch.min(surr1, surr2) + kl_coeff * kl
```
The only difference from standard PPO: there's no Critic loss to add!

</details>

In [ ]:
print("\n📝 GRPO-generated names:")
grpo_names = [generate_name(grpo_policy) for _ in range(15)]
for n in grpo_names:
    print(f"  '{n}' (reward: {reward_name(n):.2f})")

## 6. Head-to-Head: PPO vs GRPO

Let's compare PPO (with Critic) and GRPO (without Critic).
They should achieve similar performance — but GRPO uses **half the parameters!**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
ax = axes[0]
window = 10
if len(ppo_rewards) >= window:
    ppo_smooth = np.convolve(ppo_rewards, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(ppo_rewards)), ppo_smooth, color='#f43f5e', linewidth=2, label='PPO (with Critic)')
if len(grpo_rewards) >= window:
    grpo_smooth = np.convolve(grpo_rewards, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(grpo_rewards)), grpo_smooth, color='#10b981', linewidth=2, label='GRPO (no Critic!)')
ax.set_title('Training Curves')
ax.set_xlabel('Episode')
ax.set_ylabel('Average Reward')
ax.legend()

# Final reward distributions
ax = axes[1]
n_eval = 100
sft_eval = [reward_name(generate_name(sft_model)) for _ in range(n_eval)]
ppo_eval = [reward_name(generate_name(ppo_actor)) for _ in range(n_eval)]
grpo_eval = [reward_name(generate_name(grpo_policy)) for _ in range(n_eval)]

labels = ['SFT\n(baseline)', 'PPO\n(with Critic)', 'GRPO\n(no Critic!)']
data = [sft_eval, ppo_eval, grpo_eval]
colors = ['#94a3b8', '#f43f5e', '#10b981']

bp = ax.boxplot(data, labels=labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('Reward Distribution (100 samples each)')
ax.set_ylabel('Reward')

plt.tight_layout()
plt.show()

print(f"\nAvg rewards → SFT: {np.mean(sft_eval):.2f} | PPO: {np.mean(ppo_eval):.2f} | GRPO: {np.mean(grpo_eval):.2f}")

## Recap

### Key Equations

| Component | PPO | GRPO |
| :--- | :--- | :--- |
| **Advantage** | $A_t = R_t - V(s_t)$ (learned Critic) | $\hat{A}_i = \frac{r_i - \mu}{\sigma + \epsilon}$ (group average) |
| **Actor Loss** | $-\min(r_t A_t,\ \text{clip}(r_t) A_t)$ | Same! |
| **Critic Loss** | $(R_t - V(s_t))^2$ | **None!** |
| **KL Penalty** | $\beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$ | Same! |
| **Parameters** | 2× model size | 1× model size |

### What We Learned
1. **RLHF** aligns language models using RL (reward from human preferences)
2. **PPO for LLMs** is powerful but memory-hungry (needs a Critic the same size as the LLM)
3. **GRPO** replaces the learned Critic with a simple group average — half the memory, same performance
4. The secret: if you can sample multiple outputs, you don't need to *estimate* the baseline — you can *compute* it

---

### Next Up: Notebook 15 — The Great Bypass

GRPO killed the Critic. But we still need a reward model to score every output.
What if we could skip the reward model too? **DPO (Direct Preference Optimization)** does exactly that —
it collapses the entire RL pipeline into a single classification loss. 🤯